In [37]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# === Load data ===
df = pd.read_csv("general_time_error_data_bench.csv")

# === Thesis-style settings ===
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.size": 10,
    "axes.labelsize": 10,
    "legend.fontsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "figure.dpi": 300,
})

# === Figure size (inches) ===
width_in = 150 / 25.4  # 150 mm
height_in = (width_in)/2 * 0.75 + 0.5

# === Plotting setup ===
label_map = {
    'time': r'$SVD_{\mathrm{gesdd}}$',
    'time_tight_0': r'$HASVD_{\mathrm{dist}}$',
    'time_tight_1': r'$HASVD_{\mathrm{inc}}$',
    'time_tight_2': r'$2$-level dist.',
    'time_tight_3': r'$2$-level inc.',
}
markers = ['o', 's', 'd', '^', 'v']
colors = ['k', 'g', 'r', 'c', 'm']
linestyles = ['-', ':', '-.', '--', (0, (3, 1, 1, 1))]

cols = ['time', 'time_tight_0', 'time_tight_1', 'time_tight_2', 'time_tight_3']
fit_targets = ['time', 'time_tight_2']

# === Fitting function ===
def fit_polylog(x, y):
    logx = np.log(x)
    logy = np.log(y)
    slope, intercept = np.polyfit(logx, logy, 1)
    return np.exp(intercept), slope  # returns a, s for y = a * x^s

# === Filter data ===
omega = 0.1
eps1 = 1e-5
eps2 = 1e-10
df1 = df[(df["omega"] == omega) & (df["eps"] == eps1)].copy()
df2 = df[(df["omega"] == omega) & (df["eps"] == eps2)].copy()

# === Create figure ===
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(width_in, height_in))
fig.supylabel(r'Mean computational time (s)')

# === Plot: eps = 1e-5 (as scatter) ===
for i, col in enumerate(cols):
    ax1.scatter(
        df1["size"], df1[col],
        label=label_map[col],
        color=colors[i],
        marker=markers[i],
        s=30,
        zorder=3,
    )

# === Fit + plot lines for specific methods ===
for i, col in enumerate(fit_targets):
    a, s = fit_polylog(df1["size"], df1[col])
    x_fit = np.linspace(df1["size"].min(), df1["size"].max(), 200)
    y_fit = a * x_fit**s
    ax1.plot(x_fit, y_fit, linestyle='--', color=colors[i], linewidth=1.2, zorder=2)
    
    # === Annotation ===
    s_rounded = round(s, 2)
    label_text = rf"$O(n^{{{s_rounded}}})$"

    # Position text near the upper-right region of fit line
    idx = 20
    x_annot = x_fit[idx]
    if i==1:
        y_annot = y_fit[idx]-0.5
    else:
        y_annot = y_fit[idx]+5
    ax1.text(
        x_annot, y_annot,
        label_text,
        fontsize=9,
        rotation=25,
        color=colors[i],
        ha='left',
        va='bottom',

    )

ax1.set_xscale("log")
ax1.set_yscale("log")
ax1.set_xlabel(r"Size $n$")
ax1.set_title(r"$\epsilon^* = 10^{-5}$")
ax1.grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.6)

# === Plot: eps = 1e-10 ===
for i, col in enumerate(cols):
    ax2.plot(
        df2["size"], df2[col],
        label=label_map[col],
        color=colors[i],
        linestyle=linestyles[i],
        marker=markers[i],
        linewidth=1,
        markersize=4,
    )

ax2.set_xscale("log")
ax2.set_yscale("log")
ax2.set_xlabel(r"Size $n$")
ax2.set_title(r"$\epsilon^* = 10^{-10}$")
ax2.grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.6)


# === Legend ===
handles, labels = ax2.get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=3, bbox_to_anchor=(0.5, 0.99))

# === Subfigure labels ===
ax1.text(-0.15, 1.1, '(a)', transform=ax1.transAxes, fontsize=10, va='top', ha='right')
ax2.text(-0.15, 1.1, '(b)', transform=ax2.transAxes, fontsize=10, va='top', ha='right')

# === Layout and Save ===
fig.tight_layout()
fig.subplots_adjust(top=0.75)
fig.savefig("mean-computational-time-fit-annotated.pdf", format="pdf", dpi=300, transparent=True)
plt.close(fig)


In [77]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# === Load data ===
df_size = pd.read_csv("general_time_error_data_bench.csv")
df_part = pd.read_csv("general_time_partition_bench.csv")

# === Helper functions ===
def filter_df(df, omega, eps, cols):
    f = df[(df["omega"] == omega) & (df["eps"] == eps)]
    return f[cols]

def fit_polylog(x, y):
    logx = np.log10(x)
    logy = np.log10(y)
    slope, intercept = np.polyfit(logx, logy, 1)
    a = 10**intercept
    return a, slope

def annotate_panel(ax, label, xpos=0.02, ypos=0.95):
    ax.text(xpos, ypos, f'({label})', transform=ax.transAxes,
            fontsize=11, fontweight='bold', va='top', ha='left')

# === Style settings ===
label_map = {
    'time': r'$SVD_{\mathrm{gesdd}}$',
    'time_tight_0': r'$HASVD_{\mathrm{dist}}$',
    'time_tight_1': r'$HASVD_{\mathrm{inc}}$',
    'time_tight_2': r'$HASVD_{\mathrm{TLB,dist}}$',
    'time_tight_3': r'$HASVD_{\mathrm{TLB,inc}}$',
}
cols = ['time', 'time_tight_0', 'time_tight_1', 'time_tight_2', 'time_tight_3']
fit_targets = ['time', 'time_tight_2']
markers = ['o', 's', 'd', 's', 'd', '*']
colors = ['k', 'g', 'r', 'c', 'm', 'k']
linestyles = ['-', ':', ':', '-.', '-.', '--']

# === Setup figure ===
width_in = 150 / 25.4  # 150 mm
height_in = (width_in) * 0.75 + 0.6
fig, axs = plt.subplots(2, 2, figsize=(width_in, height_in), constrained_layout=True)
axs = axs.flatten()

# === Plot 1: Time vs Size, eps=1e-5 ===
df1 = filter_df(df_size, omega=0.1, eps=1e-5, cols=['size', 'eps'] + cols)
for i, col in enumerate(cols):
    axs[0].scatter(df1["size"], df1[col], label=label_map[col],
                   color=colors[i], marker=markers[i], s=30, zorder=3)

# Fit + annotate
for i, col in enumerate(fit_targets):
    a, s = fit_polylog(df1["size"], df1[col])
    x_fit = np.linspace(df1["size"].min(), df1["size"].max(), 200)
    y_fit = a * x_fit**s
    color_idx=list(label_map).index(col)
    axs[0].plot(x_fit, y_fit, linestyle='--', color=colors[color_idx], linewidth=1.2, zorder=2)
    s_rounded = round(s, 2)
    if i==0:
        y=y_fit[20]+10
    else:
        y=y_fit[20]-0.5
    axs[0].text(x_fit[20], y, rf"$\mathcal{{O}}(n^{{{s_rounded}}})$", fontsize=9,
                color=colors[color_idx], ha='left', va='bottom',)

# === Plot 2: Time vs Size, eps=1e-10 ===
df2 = filter_df(df_size, omega=0.1, eps=1e-10, cols=['size'] + cols)
for i, col in enumerate(cols):
    axs[1].plot(df2["size"], df2[col], label=label_map[col],
                color=colors[i], linestyle=linestyles[i], marker=markers[i], linewidth=1)

# === Plot 3: Time vs Partitions, eps=1e-5 ===
df3 = filter_df(df_part, omega=0.1, eps=1e-5, cols=['partitions', 'eps'] + cols)
for i, col in enumerate(cols):
    axs[2].plot(df3["partitions"], df3[col], label=label_map[col],
                color=colors[i], marker=markers[i], linestyle=linestyles[i], linewidth=1)

# === Plot 4: Time vs Partitions, eps=1e-10 ===
df4 = filter_df(df_part, omega=0.1, eps=1e-10, cols=['partitions'] + cols)
for i, col in enumerate(cols):
    axs[3].plot(df4["partitions"], df4[col], label=label_map[col],
                color=colors[i], marker=markers[i], linestyle=linestyles[i], linewidth=1)

# === Global styling for all plots ===
titles = [
    r"$\epsilon^* = 10^{-5}$ (vs. size)",
    r"$\epsilon^* = 10^{-10}$ (vs. size)",
    r"$n = 8000,\ \epsilon^* = 10^{-5}$ (vs. partitions)",
    r"$n = 8000,\ \epsilon^* = 10^{-10}$ (vs. partitions)",
]
ylabels = [
    "Time (s)",
    "Time (s)",
    "Time (s)",
    "Time (s)",
]

for i, ax in enumerate(axs):
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel("Size $n$" if i < 2 else "Number of Partitions")
    ax.set_ylabel(ylabels[i])
    ax.set_title(titles[i], fontsize=10)
    ax.grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.6)
    annotate_panel(ax, label=chr(97 + i))  # (a), (b), ...

# === Legend ===
handles, labels = axs[0].get_legend_handles_labels()
fig.legend(
    handles, labels,
    loc='lower center',
    bbox_to_anchor=(0.5, 0.02),  # push it up a bit if too low
    ncol=3,
    fontsize=8,
    frameon=False
)
fig.tight_layout(rect=[0, 0.1, 1, 1])  # leave 10% of the bottom for legend
fig.subplots_adjust(hspace=0.4, wspace=0.3)

# === Save ===
fig.savefig("mean-computational-time-4plot.pdf", format="pdf", dpi=300, transparent=True)
# fig.savefig("mean-computational-time-4plot.png", dpi=600, transparent=True)
plt.close(fig)


/tmp/ipykernel_53974/690143262.py:117: UserWarning: The figure layout has changed to tight
  fig.tight_layout(rect=[0, 0.1, 1, 1])  # leave 10% of the bottom for legend
